# Profilage DVF 2022 — Etape 2e : Surface Carrez par type de local

La Surface Carrez est l'une des trois variables les plus predictives du prix immobilier
(avec la valeur fonciere et la localisation). L'etape 1 a montre un taux de manquant
de 90,92 % sur la Surface Carrez du 1er lot, ce qui a conduit a une recommandation
prematuree d'exclusion.

Ce notebook examine le taux de remplissage de la Surface Carrez par type de local
pour determiner si ce manquant est uniforme ou s'il depend du type de bien.
La loi Carrez ne s'applique qu'aux locaux clos et couverts de plus de 8 m2 :
les parkings, caves et terrains nus ne sont pas concernes.

1. Taux de remplissage de la Surface Carrez du 1er lot par type de local
2. Taux de remplissage des 5 Surfaces Carrez par type de local
3. Comparaison Surface Carrez et Surface reelle batie pour les appartements
4. Statistiques descriptives de la Surface Carrez pour les appartements

Note technique : la Surface Carrez est stockee en VARCHAR avec la virgule comme
separateur decimal (ex. '68,08'). Les calculs numeriques convertissent d'abord
la virgule en point avant le cast en DOUBLE.

## Mode d'emploi

1. Le chemin est défini dans la **cellule 2**.
2. Executer les cellules dans l'ordre (Maj + Entree).

## Cellule 1 — Installation

In [9]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "-q"])
print("Bibliotheques pretes.")

Bibliotheques pretes.


## Cellule 2 — Reglages

**Seule cellule a modifier.**

In [10]:
import duckdb
from pathlib import Path

# Chemin a renseigner
FICHIER = Path(r"./data/dvf-2022.parquet")
SORTIE = Path(r"./figures")
SORTIE.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
pq = str(FICHIER)

assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
print(f"Fichier : {FICHIER.name}")
print(f"Lignes : {nb_lignes:,}".replace(",", " "))

# Fonction de conversion : virgule -> point -> DOUBLE
def carrez(col):
    return f'CAST(REPLACE("{col}", \',\', \'.\') AS DOUBLE)'

Fichier : dvf-2022.parquet
Lignes : 4 617 590


## Cellule 3 — Taux de remplissage de la Surface Carrez du 1er lot par type de local

La Surface Carrez est-elle bien remplie pour les appartements et absente pour les parkings ?
Ce croisement repond a la question : le manquant global de 90,92 % masque-t-il des taux
tres differents selon le type de bien ?

In [11]:
r = con.execute(f"""
    SELECT
        COALESCE(\"Type local\", '(vide)') AS type_local,
        COUNT(*) AS nb_total,
        COUNT(\"Surface Carrez du 1er lot\") AS nb_carrez,
        ROUND(100.0 * COUNT(\"Surface Carrez du 1er lot\") / COUNT(*), 1) AS pct_rempli
    FROM '{pq}'
    GROUP BY 1
    ORDER BY 4 DESC
""").fetchall()

print(f"{'Type de local':<45} {'Total':>10} {'Carrez':>10} {'% rempli':>10}")
print("-" * 78)
for typ, total, carrez_n, pct in r:
    print(f"  {typ:<43} {total:>8,} {carrez_n:>8,} {pct:>8.1f} %".replace(",", " "))

Type de local                                      Total     Carrez   % rempli
------------------------------------------------------------------------------
  Appartement                                  638 879  241 711     37.8 %
  Local industriel. commercial ou assimilé     142 535   20 506     14.4 %
  Dépendance                                  1 203 439  128 213     10.7 %
  Maison                                       756 009   14 310      1.9 %
  (vide)                                      1 876 728   14 537      0.8 %


## Cellule 4 — Taux de remplissage des 5 Surfaces Carrez par type de local

Meme analyse etendue aux 5 colonnes Surface Carrez. Le 2e lot correspond souvent
a une cave, le 3e a un parking. Leur Surface Carrez devrait etre moins souvent renseignee.

In [12]:
carrez_cols = [
    "Surface Carrez du 1er lot",
    "Surface Carrez du 2eme lot",
    "Surface Carrez du 3eme lot",
    "Surface Carrez du 4eme lot",
    "Surface Carrez du 5eme lot",
]

types = con.execute(f"""
    SELECT COALESCE(\"Type local\", '(vide)') AS t, COUNT(*) AS n
    FROM '{pq}' GROUP BY 1 ORDER BY 2 DESC
""").fetchall()

header = f"{'Type de local':<35}"
for c in carrez_cols:
    short = c.replace('Surface Carrez du ', 'C.')
    header += f" {short:>10}"
print(header)
print("-" * (35 + 11 * len(carrez_cols)))

for typ, nb in types:
    line = f"  {typ:<33}"
    for c in carrez_cols:
        pct = con.execute(f"""
            SELECT ROUND(100.0 * COUNT(\"{c}\") / COUNT(*), 1)
            FROM '{pq}'
            WHERE COALESCE(\"Type local\", '(vide)') = '{typ}'
        """).fetchone()[0]
        line += f" {pct:>9.1f} %"
    print(line)

Type de local                        C.1er lot C.2eme lot C.3eme lot C.4eme lot C.5eme lot
------------------------------------------------------------------------------------------
  (vide)                                  0.8 %       0.0 %       0.0 %       0.0 %       0.0 %
  Dépendance                             10.7 %       5.9 %       0.7 %       0.2 %       0.1 %
  Maison                                  1.9 %       0.1 %       0.0 %       0.0 %       0.0 %
  Appartement                            37.8 %      10.3 %       0.9 %       0.2 %       0.1 %
  Local industriel. commercial ou assimilé      14.4 %       2.7 %       0.8 %       0.4 %       0.2 %


## Cellule 5 — Comparaison Surface Carrez et Surface reelle batie pour les appartements

Pour les appartements qui ont les deux surfaces renseignees, quel est l'ecart ?
La Surface Carrez devrait etre legerement inferieure a la surface reelle batie
(elle exclut les murs et les surfaces sous 1,80 m).

Note : la Surface Carrez est en VARCHAR avec virgule decimale. La conversion
remplace la virgule par un point avant le cast en DOUBLE.

In [13]:
carrez_expr = carrez('Surface Carrez du 1er lot')

r = con.execute(f"""
    SELECT
        COUNT(*) AS nb,
        ROUND(MEDIAN(\"Surface reelle bati\"), 0) AS med_reelle,
        ROUND(MEDIAN({carrez_expr}), 0) AS med_carrez,
        ROUND(MEDIAN(\"Surface reelle bati\" - {carrez_expr}), 1) AS med_ecart,
        ROUND(AVG(\"Surface reelle bati\" - {carrez_expr}), 1) AS moy_ecart,
        COUNT(CASE WHEN {carrez_expr} > \"Surface reelle bati\" THEN 1 END) AS carrez_sup_reelle
    FROM '{pq}'
    WHERE \"Type local\" = 'Appartement'
        AND \"Surface Carrez du 1er lot\" IS NOT NULL
        AND \"Surface reelle bati\" IS NOT NULL
        AND \"Surface reelle bati\" > 0
""").fetchone()

print("Comparaison Surface reelle / Surface Carrez (appartements)")
print("=" * 55)
print(f"  Nb appartements avec les deux surfaces : {r[0]:,}".replace(",", " "))
print(f"  Mediane surface reelle batie    : {r[1]:,.0f} m2".replace(",", " "))
print(f"  Mediane Surface Carrez          : {r[2]:,.0f} m2".replace(",", " "))
print(f"  Ecart median (reelle - Carrez)  : {r[3]:,.1f} m2".replace(",", " "))
print(f"  Ecart moyen (reelle - Carrez)   : {r[4]:,.1f} m2".replace(",", " "))
print(f"  Cas ou Carrez > reelle          : {r[5]:,}".replace(",", " "))

Comparaison Surface reelle / Surface Carrez (appartements)
  Nb appartements avec les deux surfaces : 241 707
  Mediane surface reelle batie    : 52 m2
  Mediane Surface Carrez          : 53 m2
  Ecart median (reelle - Carrez)  : -0.1 m2
  Ecart moyen (reelle - Carrez)   : -9.1 m2
  Cas ou Carrez > reelle          : 132 345


## Cellule 6 — Statistiques descriptives de la Surface Carrez pour les appartements

Distribution de la Surface Carrez du 1er lot pour les appartements qui l'ont renseignee.

In [14]:
carrez_expr = carrez('Surface Carrez du 1er lot')

r = con.execute(f"""
    SELECT
        COUNT(*) AS nb,
        ROUND(MIN({carrez_expr}), 0) AS minimum,
        ROUND(APPROX_QUANTILE({carrez_expr}, 0.05), 0) AS P5,
        ROUND(APPROX_QUANTILE({carrez_expr}, 0.25), 0) AS Q1,
        ROUND(MEDIAN({carrez_expr}), 0) AS mediane,
        ROUND(APPROX_QUANTILE({carrez_expr}, 0.75), 0) AS Q3,
        ROUND(APPROX_QUANTILE({carrez_expr}, 0.95), 0) AS P95,
        ROUND(MAX({carrez_expr}), 0) AS maximum,
        ROUND(AVG({carrez_expr}), 0) AS moyenne
    FROM '{pq}'
    WHERE \"Type local\" = 'Appartement'
        AND \"Surface Carrez du 1er lot\" IS NOT NULL
""").fetchone()

labels = ['Nb', 'Minimum', 'P5', 'Q1', 'Mediane', 'Q3', 'P95', 'Maximum', 'Moyenne']
print("Statistiques — Surface Carrez du 1er lot (appartements, m2)")
print("=" * 55)
for label, val in zip(labels, r):
    print(f"  {label:<15} {val:>10,.0f}".replace(",", " "))

Statistiques — Surface Carrez du 1er lot (appartements, m2)
  Nb                 241 711
  Minimum                  0
  P5                      19
  Q1                      35
  Mediane                 53
  Q3                      70
  P95                    104
  Maximum              9 532
  Moyenne                 64


## Cellule 7 — Synthese de l'etape 2e

In [15]:
print("Synthese — Surface Carrez par type de local DVF 2022")
print("=" * 55)
print()

for typ, total, carrez_n, pct in con.execute(f"""
    SELECT
        COALESCE(\"Type local\", '(vide)') AS type_local,
        COUNT(*) AS nb_total,
        COUNT(\"Surface Carrez du 1er lot\") AS nb_carrez,
        ROUND(100.0 * COUNT(\"Surface Carrez du 1er lot\") / COUNT(*), 1) AS pct_rempli
    FROM '{pq}'
    GROUP BY 1
    ORDER BY 4 DESC
""").fetchall():
    print(f"  {typ:<40} {pct:>6.1f} % rempli ({carrez_n:>8,} / {total:>8,})".replace(",", " "))

Synthese — Surface Carrez par type de local DVF 2022

  Appartement                                37.8 % rempli ( 241 711 /  638 879)
  Local industriel. commercial ou assimilé   14.4 % rempli (  20 506 /  142 535)
  Dépendance                                 10.7 % rempli ( 128 213 / 1 203 439)
  Maison                                      1.9 % rempli (  14 310 /  756 009)
  (vide)                                      0.8 % rempli (  14 537 / 1 876 728)


## Cellule 8 — Fermeture

In [16]:
con.close()
print("Connexion DuckDB fermee.")

Connexion DuckDB fermee.
